# One-Electron Diagnostics for Machine-Learned Exchange-Correlation Functionals
### Rigorous Computational Verification Suite
**Author**: Dalton Omondi
**Topics**: Basis-incompleteness of same-basis UHF, PBE/r2SCAN/B3LYP delocalization error, orbital stability analysis, and symmetry-breaking bifurcation.

In [ ]:
# Cell 1: Install PySCF and dependencies
!pip install --quiet pyscf pandas tabulate
try:
    !pip install --quiet ciderpress
except Exception as e:
    print("ciderpress install note:", e)


In [ ]:
# Cell 2: Imports & Versions
import sys, os
import numpy as np
import pandas as pd
from tabulate import tabulate
import pyscf
from pyscf import gto, scf, dft
print("PySCF Version:", pyscf.__version__)
cider_available = False
try:
    import ciderpress
    print("CiderPress Version/File:", getattr(ciderpress, "__version__", ciderpress.__file__))
    cider_available = True
except ImportError as e:
    print("CiderPress not available:", e)


In [ ]:
# Cell 3: Isoelectronic Series & Basis Incompleteness (Table 4.1)
ions = [("H", 1, 0, 1), ("He+", 2, 1, 1), ("Li2+", 3, 2, 1), ("Be3+", 4, 3, 1)]
basis_list = ["def2-tzvp", "def2-svp", "def2-qzvp", "aug-cc-pvtz"]
basis_records = []
master_records = []

for basis in basis_list:
    for sym, z, charge, spin in ions:
        e_exact = -0.5 * (z**2)
        mol = gto.M(atom=f"{sym.rstrip('0123456789+-')} 0 0 0", basis=basis, charge=charge, spin=spin, verbose=0)
        mf = scf.UHF(mol)
        mf.conv_tol = 1e-12
        mf.kernel()
        incomp_mha = (mf.e_tot - e_exact) * 1000.0
        basis_records.append({"Basis": basis, "Ion": sym, "Z": z, "E_exact": e_exact, "E_UHF": mf.e_tot, "Incomp_mHa": incomp_mha, "Cycles": mf.cycles})
        master_records.append({"functional": "UHF", "system": sym, "basis": basis, "R": np.nan, "E": mf.e_tot, "converged": mf.converged, "cycles": mf.cycles, "stable": True, "q_Ha": 1.0 - charge, "q_Hb": np.nan, "notes": "Exact UHF reference"})

df_basis = pd.DataFrame(basis_records)
print(tabulate(df_basis, headers="keys", tablefmt="github", floatfmt=".6f"))


In [ ]:
# Cell 4: Isoelectronic Series DFT SIE (Table 4.2 & 4.3)
dft_funcs = ["PBE", "r2SCAN", "SCAN", "B3LYP"]
primary_basis = "def2-tzvp"
iso_dft_records = []

for xc in dft_funcs:
    for sym, z, charge, spin in ions:
        e_exact = -0.5 * (z**2)
        mol = gto.M(atom=f"{sym.rstrip('0123456789+-')} 0 0 0", basis=primary_basis, charge=charge, spin=spin, verbose=0)
        mf_ref = scf.UHF(mol)
        mf_ref.conv_tol = 1e-12
        mf_ref.kernel()
        e_uhf = mf_ref.e_tot

        mf = dft.UKS(mol)
        mf.xc = xc
        mf.conv_tol = 1e-11
        mf.kernel()
        e_dft = mf.e_tot
        is_stable = True  # One-electron atomic cations (N=1) are spherical and unconditionally stable
        naive_sie = (e_dft - e_exact) * 1000.0
        same_sie = (e_dft - e_uhf) * 1000.0
        ratio = naive_sie / same_sie if abs(same_sie) > 1e-6 else np.nan
        iso_dft_records.append({"Functional": xc, "Ion": sym, "Z": z, "E_DFT": e_dft, "E_UHF": e_uhf, "Naive_SIE_mHa": naive_sie, "Same_SIE_mHa": same_sie, "Inflation_Ratio": ratio, "Stable": is_stable})
        master_records.append({"functional": xc, "system": sym, "basis": primary_basis, "R": np.nan, "E": e_dft, "converged": mf.converged, "cycles": mf.cycles, "stable": is_stable, "q_Ha": 1.0 - charge, "q_Hb": np.nan, "notes": f"Naive SIE: {naive_sie:.3f} mHa, Same: {same_sie:.3f} mHa"})

df_iso_dft = pd.DataFrame(iso_dft_records)
print(tabulate(df_iso_dft, headers="keys", tablefmt="github", floatfmt=".4f"))


In [ ]:
# Cell 5: Stretched H2+ Dissociation Scan & Physical Bifurcation Analysis
r_grid = [0.5, 0.75, 1.06, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 10.0, 15.0]
h2_records = []

for xc in ["UHF", "PBE", "r2SCAN", "B3LYP"]:
    print(f"=== Functional: {xc} ===")
    for r in r_grid:
        mol = gto.M(atom=f"H 0 0 0; H 0 0 {r}", basis=primary_basis, charge=1, spin=1, verbose=0)
        # Exact UHF reference for this R
        mf_ref = scf.UHF(mol)
        mf_ref.conv_tol = 1e-12
        mf_ref.kernel()
        e_ref = mf_ref.e_tot

        if xc == "UHF":
            e_calc = e_ref
            charges = [0.5, 0.5]
            is_stable = True
            conv = True
            cycles = mf_ref.cycles
            e_broken = np.nan
            q_broken = (np.nan, np.nan)
        else:
            # 1. Symmetric initial guess / default solution
            mf = dft.UKS(mol)
            mf.xc = xc
            mf.conv_tol = 1e-10
            mf.max_cycle = 200
            mf.kernel()
            e_calc = mf.e_tot
            conv = mf.converged
            cycles = mf.cycles
            charges = mf.mulliken_pop(verbose=0)[1]

            # 2. Broken-symmetry initial guess (seed electron on atom 0)
            e_broken = np.nan
            q_broken = (np.nan, np.nan)
            try:
                mf_bs = dft.UKS(mol)
                mf_bs.xc = xc
                mf_bs.conv_tol = 1e-10
                mf_bs.max_cycle = 200
                dm_bs = mf.make_rdm1()
                ao_slices = mol.aoslice_by_atom()
                a1_start, a1_end = ao_slices[1, 2], ao_slices[1, 3]
                dm_bs[0, a1_start:a1_end, :] = 0.0
                dm_bs[0, :, a1_start:a1_end] = 0.0
                s = mol.intor("int1e_ovlp")
                tr = float(np.einsum("ij,ji->", dm_bs[0], s))
                if tr > 1e-6:
                    dm_bs[0] /= tr
                mf_bs.kernel(dm0=dm_bs)
                if mf_bs.converged:
                    e_broken = mf_bs.e_tot
                    q_broken = mf_bs.mulliken_pop(verbose=0)[1]
            except Exception as e_bs:
                pass

            # Physical orbital stability: does a broken-symmetry state have strictly lower energy?
            if not np.isnan(e_broken) and (e_broken < e_calc - 1e-5) and abs(q_broken[0] - q_broken[1]) > 0.1:
                is_stable = False  # Symmetric state is an unstable saddle point
            else:
                is_stable = True   # Symmetric state is the stable ground state

        delta_mha = (e_calc - e_ref) * 1000.0
        h2_records.append({"Functional": xc, "R": r, "E": e_calc, "E_UHF": e_ref, "Delta_mHa": delta_mha, "qa": charges[0], "qb": charges[1], "Conv": conv, "Cycles": cycles, "Stable": is_stable, "E_broken": e_broken, "qa_broken": q_broken[0], "qb_broken": q_broken[1]})
        master_records.append({"functional": xc, "system": "H2+", "basis": primary_basis, "R": r, "E": e_calc, "converged": conv, "cycles": cycles, "stable": is_stable, "q_Ha": charges[0], "q_Hb": charges[1], "notes": f"Delta: {delta_mha:.3f} mHa. E_broken: {e_broken}"})
        print(f"  R={r:5.2f} A | E_sym: {e_calc:.6f} Ha | E_brk: {e_broken:.6f} Ha | Delta: {delta_mha:+8.3f} mHa | qa={charges[0]:.3f}, qb={charges[1]:.3f} | Stable_sym: {is_stable} | Conv: {conv}")

df_h2 = pd.DataFrame(h2_records)


In [ ]:
# Cell 6: Export Master CSV
df_master = pd.DataFrame(master_records)
df_master.to_csv("master_dft_one_electron_dataset.csv", index=False)
print("Exported master CSV with", len(df_master), "rows.")

# Summary check of B3LYP anomaly
b3_r10 = df_h2[(df_h2["Functional"] == "B3LYP") & (df_h2["R"] == 10.0)]
print("\n=== B3LYP AT R = 10 A SUMMARY ===")
print(tabulate(b3_r10, headers="keys", tablefmt="github"))
